# Integrated Gradients

Integrated Gradients is an axiomatic attribution method that assigns importance scores to input features. It integrates the gradients along a straight-line path from a baseline (e.g., zero input) to the actual input. It satisfies two key axioms: Sensitivity and Implementation Invariance.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

# Build a simple Keras Sequential model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

# Compile and fit
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=100, verbose=0)

In [ ]:
def integrated_gradients(model, input_sample, baseline, num_steps=50):
    """
    Compute Integrated Gradients attribution for a given input sample.
    
    Args:
        model: A TensorFlow Keras model
        input_sample: Input sample as a numpy array (shape: (features,))
        baseline: Baseline input, typically zeros (shape: (features,))
        num_steps: Number of interpolation steps
    
    Returns:
        attributions: Array of attribution scores per feature
    """
    # Ensure inputs are tensors
    input_sample = tf.convert_to_tensor(input_sample, dtype=tf.float32)
    baseline = tf.convert_to_tensor(baseline, dtype=tf.float32)
    
    # Initialize cumulative gradients
    accumulated_grads = None
    
    # Create interpolated inputs and compute gradients
    for step in range(num_steps):
        # Interpolation parameter
        alpha = step / num_steps
        
        # Interpolated input
        interpolated_input = baseline + alpha * (input_sample - baseline)
        interpolated_input = tf.expand_dims(interpolated_input, axis=0)
        
        # Compute gradients
        with tf.GradientTape() as tape:
            tape.watch(interpolated_input)
            logits = model(interpolated_input)
            # Use the predicted class for attribution
            pred_class = tf.argmax(logits, axis=1)[0]
            output = logits[0, pred_class]
        
        gradients = tape.gradient(output, interpolated_input)
        
        if accumulated_grads is None:
            accumulated_grads = gradients
        else:
            accumulated_grads += gradients
    
    # Average gradients
    avg_grads = accumulated_grads / num_steps
    
    # Compute attributions: (input - baseline) * avg_gradients
    attributions = (input_sample - baseline) * tf.squeeze(avg_grads)
    
    return attributions.numpy()

In [ ]:
# Pick a test sample
test_sample = X_test[0]
baseline = np.zeros_like(test_sample)

# Compute integrated gradients
attributions = integrated_gradients(model, test_sample, baseline, num_steps=50)

# Feature names from Iris dataset
feature_names = iris.feature_names

# Plot bar chart
plt.figure(figsize=(10, 6))
plt.bar(feature_names, attributions)
plt.xlabel('Features')
plt.ylabel('Attribution Score')
plt.title('Integrated Gradients Attribution Scores')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Attributions:', attributions)

## Interpretation

The bars in the chart represent the attribution score for each feature, indicating how much that feature contributed to the model's prediction for this particular input sample. According to the completeness axiom, the sum of all attribution scores approximates the difference between the model output at the actual input and the model output at the baseline (zero input), ensuring that all prediction changes are fully accounted for by the attributed features.